# 02 — Preprocessing & SOSA Mapping

**Goal:** Clean the 3W dataset and map it to the SOSA/SSN ontology for TKG ingestion.

## Steps:
1. Load and merge all instances
2. Handle NaN and frozen variables
3. Align timestamps (clock skew)
4. Map sensors to SOSA vocabulary
5. Add bitemporal annotation (valid_time + transaction_time)
6. Export preprocessed data for Neo4j loader

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT   = Path('../../data/UseCase2/3w_dataset')
OUTPUT_DIR  = Path('../../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SENSORS = ['P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-JUS-CKGL','QGL']

# SOSA mapping (Decision 1 — justified by Haller et al. [16])
SOSA_MAPPING = {
    'P-PDG':      {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'downhole'},
    'P-TPT':      {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'subsea_christmas_tree'},
    'T-TPT':      {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'subsea_christmas_tree'},
    'P-MON-CKP':  {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'platform_upstream_pck'},
    'T-JUS-CKP':  {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'platform_downstream_pck'},
    'P-JUS-CKGL': {'observable_property': 'Pressure', 'unit': 'bar', 'position': 'gas_lift_downstream'},
    'T-JUS-CKGL': {'observable_property': 'Temperature', 'unit': 'celsius', 'position': 'gas_lift_downstream'},
    'QGL':        {'observable_property': 'VolumetricFlowRate', 'unit': 'sm3_per_s', 'position': 'gas_lift_injection'},
}

# Physical order for PRECEDES relation (Decision 1)
# Justified by Vargas et al. (2019): PDG(downhole) → TPT(subsea) → PCK(surface)
SENSOR_ORDER = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'T-JUS-CKGL', 'QGL']

EVENT_TYPES = {0:'Normal',1:'Abrupt_BSW',2:'Spurious_DHSV',3:'Severe_Slugging',
               4:'Flow_Instability',5:'Rapid_Productivity_Loss',
               6:'Quick_Restriction_PCK',7:'Scaling_PCK',8:'Hydrate'}

print('✅ Config loaded')
print(f'   Output dir: {OUTPUT_DIR}')

## 1. Load All Instances

In [ ]:
def load_all_instances(data_root: Path) -> pd.DataFrame:
    """
    Carica tutte le istanze CSV del 3W dataset.
    Aggiunge: well_id, event_type, instance_id da filename.
    """
    if not data_root.exists():
        print('⚠️  Dataset non trovato — esegui prima il download')
        return pd.DataFrame()

    dfs = []
    for event_dir in sorted(data_root.iterdir()):
        if not event_dir.is_dir():
            continue
        event_code = int(event_dir.name)
        for csv_file in sorted(event_dir.glob('*.csv')):
            try:
                df = pd.read_csv(csv_file, index_col='timestamp', parse_dates=True)
                df['well_id']     = csv_file.stem
                df['event_type']  = event_code
                df['instance_id'] = csv_file.stem
                dfs.append(df)
            except Exception as e:
                print(f'  ⚠️  Error loading {csv_file.name}: {e}')

    if not dfs:
        return pd.DataFrame()

    combined = pd.concat(dfs).reset_index()
    combined = combined.rename(columns={'index': 'timestamp'})
    print(f'✅ Loaded {len(dfs)} instances, {len(combined):,} total observations')
    return combined

df_raw = load_all_instances(DATA_ROOT)
if not df_raw.empty:
    print(df_raw.head(3))

## 2. Handle NaN and Frozen Variables

In [ ]:
def flag_data_quality(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flags NaN and frozen variables — does NOT impute.
    
    Design decision (TKG):
    - NaN: flagged with quality_flag='missing', inferred from graph neighbors
      Ref: Khan et al. [9] — graph structure provides redundant evidence paths
    - Frozen: flagged with quality_flag='frozen', validated by SHACL
      Ref: SHACL validation (TKG Report Section 2.4)
    """
    df = df.copy()
    df['quality_flag'] = 'ok'

    for sensor in SENSORS:
        if sensor not in df.columns:
            continue
        # Flag NaN
        nan_mask = df[sensor].isnull()
        df.loc[nan_mask, 'quality_flag'] = 'missing'

        # Flag frozen per instance (same value throughout)
        for instance_id, group in df.groupby('instance_id'):
            if group[sensor].nunique() == 1 and not group[sensor].isnull().all():
                df.loc[group.index, 'quality_flag'] = 'frozen'

    quality_counts = df['quality_flag'].value_counts()
    print('Data quality flags:')
    print(quality_counts.to_string())
    return df

if not df_raw.empty:
    df_flagged = flag_data_quality(df_raw)
else:
    print('⚠️  Load dataset first')

## 3. Add Bitemporal Annotation

In [ ]:
def add_bitemporality(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds valid_time and transaction_time to every observation.

    valid_time    = when the sensor reading is true in the real world
    tx_time       = when it was ingested into the TKG

    Ref: Rost et al. [22] — bitemporal property graph model
    """
    df = df.copy()
    df['valid_from'] = pd.to_datetime(df['timestamp']).dt.strftime('%Y-%m-%dT%H:%M:%S')
    df['valid_to']   = None   # null = still valid
    df['tx_time']    = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S')
    print('✅ Bitemporal annotation added')
    print(f'   valid_from range: {df["valid_from"].min()} → {df["valid_from"].max()}')
    return df

if not df_raw.empty:
    df_bitemporal = add_bitemporality(df_flagged)
else:
    print('⚠️  Load dataset first')

## 4. Export Preprocessed Data

In [ ]:
def export_for_neo4j(df: pd.DataFrame, output_dir: Path):
    """
    Exports preprocessed data in format ready for Neo4j loader.
    Saves: observations.parquet + sosa_mapping.json
    """
    # Save observations
    output_path = output_dir / 'observations_3w.parquet'
    df.to_parquet(output_path, index=False)
    print(f'✅ Observations saved: {output_path}')
    print(f'   Shape: {df.shape}')

    # Save SOSA mapping
    mapping_path = output_dir / 'sosa_mapping.json'
    with open(mapping_path, 'w') as f:
        json.dump(SOSA_MAPPING, f, indent=2)
    print(f'✅ SOSA mapping saved: {mapping_path}')

    # Save sensor order for PRECEDES relation
    order_path = output_dir / 'sensor_order.json'
    with open(order_path, 'w') as f:
        json.dump({'precedes_order': SENSOR_ORDER,
                   'justification': 'Physical order: downhole → subsea → surface (Vargas et al. 2019)'}, f, indent=2)
    print(f'✅ Sensor order saved: {order_path}')

if not df_raw.empty:
    export_for_neo4j(df_bitemporal, OUTPUT_DIR)
else:
    print('⚠️  Load dataset first')